# RK1828 Qwen3-ForcedAligner 强制对齐 Demo


[Qwen3-ForcedAligner](https://github.com/QwenLM/Qwen3-ForcedAligner)：
给定**音频 + 已知文本**，输出每个字/词的时间戳（TSV），中英文均可，实测 RTF 0.041。
典型用途：字幕生成、歌词对齐、语音数据标注。

| 步骤 | 内容 |
|---|---|
| 1 | 下载模型和示例音频（可选 HF / HF-Mirror 源） |
| 2 | 解压预编译 demo |
| 3 | 输入文本并运行 |
| 4 | 查看时间戳结果 |

## 0. 环境检查

先确认 RKNN3 运行库（`librknn3_api.so`）和 RK1828 设备都正常。
如失败，请先安装驱动/运行时（`rknn3-rk182x` DEB 包），并用 `sudo rknn-smi info` 确认能看到 RK1828。

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # 加载 /usr/lib/librknn3_api.so；失败说明 Runtime 未安装

devices = find_devices()
if not devices:
    raise RuntimeError("未发现 RK1828 设备！请检查: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"发现设备: id={dev_id}  接口={dev_type}")

## 1. 下载模型

本 demo 的模型文件托管在 Hugging Face（`qhddz` 账号）。
**先用下拉框选择下载源**，再运行下一个单元格开始下载：

- **HF-Mirror 国内镜像**：`hf-mirror.com`，国内直连，推荐
- **Hugging Face 官方**：`huggingface.co`，海外或已有代理时用

> 已存在（且非空）的文件会**自动跳过**，重复运行不会重新下载。

In [ ]:
from nb_download import source_picker

# (HF仓库名, 仓库内路径, 本地路径)
FILES = [
    ("qwen3-forced-aligner-rknn3-rk1828", "audio.wav", "demos/forced_aligner/audio.wav"),
    ("qwen3-forced-aligner-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/forced_aligner/demo_x86_64.tar.gz"),
    ("qwen3-forced-aligner-rknn3-rk1828", "mel_128_filters.txt", "demos/forced_aligner/mel_128_filters.txt"),
    ("qwen3-forced-aligner-rknn3-rk1828", "model/Qwen3-ForcedAligner-0.6B_decoder.embed.bin", "demos/forced_aligner/model/Qwen3-ForcedAligner-0.6B_decoder.embed.bin"),
    ("qwen3-forced-aligner-rknn3-rk1828", "model/Qwen3-ForcedAligner-0.6B_decoder.tokenizer.gguf", "demos/forced_aligner/model/Qwen3-ForcedAligner-0.6B_decoder.tokenizer.gguf"),
    ("qwen3-forced-aligner-rknn3-rk1828", "model/Qwen3-ForcedAligner-0.6B_decoder_config.txt", "demos/forced_aligner/model/Qwen3-ForcedAligner-0.6B_decoder_config.txt"),
    ("qwen3-forced-aligner-rknn3-rk1828", "model/Qwen3-ForcedAligner-0.6B_decoder_rules.bin", "demos/forced_aligner/model/Qwen3-ForcedAligner-0.6B_decoder_rules.bin"),
    ("qwen3-forced-aligner-rknn3-rk1828", "model/decoder.rknn", "demos/forced_aligner/model/decoder.rknn"),
    ("qwen3-forced-aligner-rknn3-rk1828", "model/decoder.weight", "demos/forced_aligner/model/decoder.weight"),
    ("qwen3-forced-aligner-rknn3-rk1828", "model/encoder.rknn", "demos/forced_aligner/model/encoder.rknn"),
    ("qwen3-forced-aligner-rknn3-rk1828", "model/encoder.weight", "demos/forced_aligner/model/encoder.weight"),
]

w_source = source_picker()

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, w_source.value)

## 2. 解压预编译 demo

`demo_x86_64.tar.gz` 里是官方流程编译好的 x86_64 C++ demo 和运行时库（`lib/`）。
HF 不允许直接托管可执行文件，所以打包成 tar.gz，解压一次即可（重复运行自动跳过）。

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/forced_aligner/demo_x86_64.tar.gz", "demos/forced_aligner")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. 输入要对齐的文本

仓库自带示例 `audio.wav`（中文："欢迎大家来到么哒社区进行体验"），
也可以换自己的 wav + 对应文本。

In [ ]:
import ipywidgets as widgets
from IPython.display import display

w_wav = widgets.Text(value="audio.wav", description="音频:",
                     layout=widgets.Layout(width="60%"))
w_text = widgets.Textarea(value="欢迎大家来到么哒社区进行体验",
                          description="文本:",
                          layout=widgets.Layout(width="70%", height="50px"))
w_lang = widgets.Dropdown(options=["Chinese", "English"], description="语言:")
display(w_wav, w_text, w_lang)
print("选好后继续往下运行 ▼")

In [ ]:
DEMO = "demos/forced_aligner"
before = snapshot(DEMO)
run_demo(["./rknn_qwen3_forced_aligner_demo",
          "--model_path", "./model",
          "--audio_path", w_wav.value,
          "--text", w_text.value,
          "--language", w_lang.value,
          "--mel_filters_path", "./mel_128_filters.txt",
          "--output_path", "./result.tsv"], DEMO)

## 4. 查看时间戳（TSV：开始秒 / 结束秒 / 字）

In [ ]:
import csv

rows = list(csv.reader(open("demos/forced_aligner/result.tsv"), delimiter="\t"))
if rows and rows[0][0].lower().startswith("start"):
    rows = rows[1:]  # 去掉表头 start_ms / end_ms / text
rows = [(float(r[0]) / 1000, float(r[1]) / 1000, r[2]) for r in rows]  # ms -> s
print(f"{'开始(s)':>8} {'结束(s)':>8}  字")
for s, e, t in rows:
    print(f"{s:8.2f} {e:8.2f}  {t}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf

wav_path = "demos/forced_aligner/audio.wav"
data, sr = sf.read(wav_path)
t = np.arange(len(data)) / sr
plt.figure(figsize=(12, 3))
plt.plot(t, data, lw=0.3, color="#888")
for s, e, txt in rows:
    plt.axvspan(s, e, alpha=0.25)
    plt.text((s + e) / 2, 0.8, txt, ha="center", fontsize=9)
plt.title("音频波形上的对齐结果")
plt.xlabel("秒")
plt.tight_layout()
plt.show()

## 小结

- 对齐不是"识别"：文本已知，模型只负责把每个字**钉到时间轴上**。
- 波形图上可以直观看到每段 span 与发音的对应关系。